In [1]:
import pandas as pd
import numpy as np
from scipy.signal import argrelextrema
from scipy.signal import find_peaks
from itertools import product
from tqdm import tqdm
import time
import numba as nb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm import tqdm

In [18]:
df=pd.read_csv('../../all_data_EUR_USD.csv')

In [19]:
df

,time,o,h,l,c,volume,complete
0,2005-01-02 18:15:00,1.35600,1.35600,1.35600,1.35600,1,True
1,2005-01-02 18:30:00,1.35600,1.35600,1.35600,1.35600,1,True
2,2005-01-02 18:45:00,1.35670,1.35680,1.35650,1.35670,4,True
3,2005-01-02 19:00:00,1.35690,1.35700,1.35690,1.35690,5,True
4,2005-01-02 19:15:00,1.35650,1.35690,1.35560,1.35560,27,True
...,...,...,...,...,...,...,...
432767,2023-03-03 20:45:00,1.06330,1.06352,1.06306,1.06352,983,True
432768,2023-03-03 21:00:00,1.06350,1.06351,1.06325,1.06342,343,True
432769,2023-03-03 21:15:00,1.06346,1.06346,1.06321,1.06322,212,True
432770,2023-03-03 21:30:00,1.06322,1.06322,1.06303,1.06319,141,True


In [20]:
df.set_index('time',inplace=True)

In [21]:
def calc_hh(high_vals):

    temp_max=high_vals[0]
    
    for i in range(len(high_vals)):

        if high_vals[i]>=temp_max:
            temp_max=high_vals[i]
            

    if temp_max==high_vals[len(high_vals)-1]:

        return temp_max
    else:

        return np.nan

In [22]:
def calc_ll(low_vals):

    temp_min=low_vals[0]
    
    for i in range(len(low_vals)):

        if low_vals[i]<=temp_min:
            temp_min=low_vals[i]
            

    if temp_min==low_vals[len(low_vals)-1]:

        return temp_min
    else:

        return np.nan

In [23]:
class Fibonacci():

    def __init__(self, data):

        self.data=data

        self.params_range={'retracement':[0.236, 0.382,0.500,0.618, 0.786],
                          'period':[1,5,10,14,28,40,50],}

        
        self.possible_strats={'strategy_desc':'Fibonacci Retracements',
                              'p_a_b_retr':{'name': 'price above below retracement level',
                                           'positions':{'buy':'price above retracement level',
                                                       'sell':'price below retracement level'},
                                            'pos_columns':{}}}
                            
    def create_params_combs(self):

        return list(product(*self.params_range.values()))

        
    def calc_indicator(self, retracement,period):

        self.fib_colname=f'retr_{retracement}_per_{period}'
        self.trend_colname=f'pos_trend_{self.fib_colname}'
        self.retracement=retracement
        self.period=period
        df_fib=self.data.copy()

        df_fib['hh']=df_fib['h'].rolling(period+1, min_periods=1).apply(calc_hh, raw=True, engine='numba')
        df_fib['ll']=df_fib['l'].rolling(period+1, min_periods=1).apply(calc_ll, raw=True, engine='numba')
        df_fib['hh_date']=np.nan
        df_fib['hh_date']=np.where(df_fib['hh'].notna(),df_fib.index,df_fib['hh_date'])
        df_fib['ll_date']=np.nan
        df_fib['ll_date']=np.where(df_fib['ll'].notna(),df_fib.index,df_fib['ll_date'])
        df_fib['hh']=df_fib['hh'].ffill()
        df_fib['ll']=df_fib['ll'].ffill()
        df_fib['hh_date']=df_fib['hh_date'].ffill()
        df_fib['ll_date']=df_fib['ll_date'].ffill()

        df_fib['Trend'] = np.where(df_fib.hh_date > df_fib.ll_date, "Up", "Down")
        df_fib[self.trend_colname]=np.nan

        df_fib[self.trend_colname]=df_fib['Trend'].map({'Up':1,'Down':-1})
        self.pos_ch_trend_colname=f'pos_ch_trend_{self.fib_colname}'
        df_fib[self.pos_ch_trend_colname]=np.nan
        df_fib[self.pos_ch_trend_colname]=np.where(df_fib[self.trend_colname]!=df_fib[self.trend_colname].shift(),df_fib[self.trend_colname], df_fib[self.pos_ch_trend_colname])

        

        

        df_fib[self.fib_colname] = np.where(df_fib[self.trend_colname] == "Up", df_fib.hh - (df_fib.hh-df_fib.ll) * retracement, df_fib.hh - (df_fib.hh-df_fib.ll) * (1-retracement))
        
        self.data=df_fib

    def calc_position(self):
        df_pos=self.data.copy()
        for k in self.possible_strats.keys():
            if k=='p_a_b_retr':

                
                pos_ch_colname=f'pos_ch_retr_{self.retracement}_per_{self.period}'
                pos_colname=f'pos_retr_{self.retracement}_per_{self.period}'
        
                df_pos[pos_ch_colname]=np.nan
        
                df_pos[pos_ch_colname]=np.where((df_pos['c']<=df_pos[self.fib_colname])&(df_pos['c'].shift()>df_pos[self.fib_colname].shift()),-1,df_pos[pos_ch_colname])
                df_pos[pos_ch_colname]=np.where((df_pos['c']>=df_pos[self.fib_colname])&(df_pos['c'].shift()<df_pos[self.fib_colname].shift()),1,df_pos[pos_ch_colname])
                df_pos[pos_colname]=df_pos[pos_ch_colname].ffill()
        self.data=df_pos[[col for col in df_pos.columns if col not in ['hh','hh_date','ll','ll_date','Trend']]]

    def plot_pos_chart(self, n_candles:int=400):

        col_pos_to_plot=None
        
        for c in self.data.columns:

            if 'pos_ch' in c:
                col_pos_to_plot=c
                break

        df_plot=self.data.iloc[:n_candles].copy()
        

        

        y_color_sell=df_plot[df_plot[ col_pos_to_plot]==-1]['h']*1.0002
        y_color_sell_index=df_plot[df_plot[ col_pos_to_plot]==-1].index
        y_color_buy=df_plot[df_plot[ col_pos_to_plot]==1]['l']*0.9998
        y_color_buy_index=df_plot[df_plot[ col_pos_to_plot]==1].index

    
       
        figure = make_subplots(rows=2, cols=1, row_heights=[0.7,0.3], shared_xaxes=True,vertical_spacing=0.01)
        figure.update_layout(height=800, width=1200, title_text=col_pos_to_plot)
        
    
        figure.add_trace(go.Candlestick(x=df_plot.index,
                                        open=df_plot['o'],
                                        high=df_plot['h'],
                                        low=df_plot['l'],
                                        close=df_plot['c'],
                                        name='price'), row=1, col=1)
        
        figure.add_trace(go.Scatter(x=df_plot.index,y=df_plot["R50"],mode='lines',line_color='yellow',name="R50"),col=1,row=1 )
        figure.add_trace(go.Scatter(x=y_color_sell_index, y=y_color_sell, mode='markers', marker_symbol='arrow-down', marker_color='red', name='sell', marker_size=10), col=1, row=1)
        figure.add_trace(go.Scatter(x=y_color_buy_index, y=y_color_buy, mode='markers', marker_symbol='arrow-up', marker_color='green', name='buy', marker_size=10), col=1, row=1)

        
       
    
        figure.update_layout(xaxis_rangeslider_visible=False)
        figure.update_xaxes(
        rangebreaks=[
            dict(bounds=["sat", "mon"])]
    )
        figure.show()

        


                
        
        

In [24]:
fib=Fibonacci(df)

In [25]:
params=fib.create_params_combs()

In [26]:
params

[(0.236, 1),
 (0.236, 5),
 (0.236, 10),
 (0.236, 14),
 (0.236, 28),
 (0.236, 40),
 (0.236, 50),
 (0.382, 1),
 (0.382, 5),
 (0.382, 10),
 (0.382, 14),
 (0.382, 28),
 (0.382, 40),
 (0.382, 50),
 (0.5, 1),
 (0.5, 5),
 (0.5, 10),
 (0.5, 14),
 (0.5, 28),
 (0.5, 40),
 (0.5, 50),
 (0.618, 1),
 (0.618, 5),
 (0.618, 10),
 (0.618, 14),
 (0.618, 28),
 (0.618, 40),
 (0.618, 50),
 (0.786, 1),
 (0.786, 5),
 (0.786, 10),
 (0.786, 14),
 (0.786, 28),
 (0.786, 40),
 (0.786, 50)]

In [28]:
for p in tqdm(params):

    fib.calc_indicator(*p)
    fib.calc_position()

100%|██████████| 35/35 [02:11<00:00,  3.76s/it]


In [29]:
fib.data

,o,h,l,c,volume,complete,pos_trend_retr_0.236_per_1,pos_ch_trend_retr_0.236_per_1,retr_0.236_per_1,pos_ch_retr_0.236_per_1,...,pos_trend_retr_0.786_per_40,pos_ch_trend_retr_0.786_per_40,retr_0.786_per_40,pos_ch_retr_0.786_per_40,pos_retr_0.786_per_40,pos_trend_retr_0.786_per_50,pos_ch_trend_retr_0.786_per_50,retr_0.786_per_50,pos_ch_retr_0.786_per_50,pos_retr_0.786_per_50
time,,,,,,,,,,,,,,,,,,,,,
2005-01-02 18:15:00,1.35600,1.35600,1.35600,1.35600,1,True,-1,-1.0,1.356000,NaN,...,-1,-1.0,1.356000,NaN,NaN,-1,-1.0,1.356000,NaN,NaN
2005-01-02 18:30:00,1.35600,1.35600,1.35600,1.35600,1,True,-1,NaN,1.356000,NaN,...,-1,NaN,1.356000,NaN,NaN,-1,NaN,1.356000,NaN,NaN
2005-01-02 18:45:00,1.35670,1.35680,1.35650,1.35670,4,True,1,1.0,1.356189,NaN,...,1,1.0,1.356629,NaN,NaN,1,1.0,1.356629,NaN,NaN
2005-01-02 19:00:00,1.35690,1.35700,1.35690,1.35690,5,True,1,NaN,1.356236,NaN,...,1,NaN,1.356786,NaN,NaN,1,NaN,1.356786,NaN,NaN
2005-01-02 19:15:00,1.35650,1.35690,1.35560,1.35560,27,True,-1,-1.0,1.355930,-1.0,...,-1,-1.0,1.356700,-1.0,-1.0,-1,-1.0,1.356700,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-03-03 20:45:00,1.06330,1.06352,1.06306,1.06352,983,True,-1,NaN,1.063254,1.0,...,1,NaN,1.062799,NaN,1.0,1,NaN,1.062799,NaN,1.0
2023-03-03 21:00:00,1.06350,1.06351,1.06325,1.06342,343,True,-1,NaN,1.063254,NaN,...,1,NaN,1.062799,NaN,1.0,1,NaN,1.062799,NaN,1.0
2023-03-03 21:15:00,1.06346,1.06346,1.06321,1.06322,212,True,-1,NaN,1.063368,-1.0,...,1,NaN,1.062799,NaN,1.0,1,NaN,1.062799,NaN,1.0


In [30]:
fib.data.columns

Index(['o', 'h', 'l', 'c', 'volume', 'complete', 'pos_trend_retr_0.236_per_1',
       'pos_ch_trend_retr_0.236_per_1', 'retr_0.236_per_1',
       'pos_ch_retr_0.236_per_1',
       ...
       'pos_trend_retr_0.786_per_40', 'pos_ch_trend_retr_0.786_per_40',
       'retr_0.786_per_40', 'pos_ch_retr_0.786_per_40',
       'pos_retr_0.786_per_40', 'pos_trend_retr_0.786_per_50',
       'pos_ch_trend_retr_0.786_per_50', 'retr_0.786_per_50',
       'pos_ch_retr_0.786_per_50', 'pos_retr_0.786_per_50'],
      dtype='object', length=181)

In [31]:
fib.data.to_csv('Fibonacci_data.csv')

In [32]:
with open('fibonacci.json', "w") as f:
    json.dump(fib.possible_strats, f)

In [ ]:
fib.data[:50]

In [ ]:
df_plot=fib.data.iloc[:400].copy()
        

        

y_color_sell=df_plot[df_plot['pos_ch_retr_0.382_per_20']==-1]['h']*1.0002
y_color_sell_index=df_plot[df_plot['pos_ch_retr_0.382_per_20']==-1].index
y_color_buy=df_plot[df_plot['pos_ch_retr_0.382_per_20']==1]['l']*0.9998
y_color_buy_index=df_plot[df_plot['pos_ch_retr_0.382_per_20']==1].index



figure = make_subplots(rows=2, cols=1, row_heights=[0.7,0.3], shared_xaxes=True,vertical_spacing=0.01)
figure.update_layout(height=800, width=1200, title_text='pos_ch_retr_0.382_per_20')


figure.add_trace(go.Candlestick(x=df_plot.index,
                                open=df_plot['o'],
                                high=df_plot['h'],
                                low=df_plot['l'],
                                close=df_plot['c'],
                                name='price'), row=1, col=1)

figure.add_trace(go.Scatter(x=df_plot.index,y=df_plot["retr_0.236_per_20"],mode='lines',line_color='yellow',name="retr_0.236_per_20"),col=1,row=1 )
figure.add_trace(go.Scatter(x=df_plot.index,y=df_plot["retr_0.382_per_20"],mode='lines',line_color='black',name="retr_0.382_per_20"),col=1,row=1 )
figure.add_trace(go.Scatter(x=y_color_sell_index, y=y_color_sell, mode='markers', marker_symbol='arrow-down', marker_color='red', name='sell', marker_size=10), col=1, row=1)
figure.add_trace(go.Scatter(x=y_color_buy_index, y=y_color_buy, mode='markers', marker_symbol='arrow-up', marker_color='green', name='buy', marker_size=10), col=1, row=1)




figure.update_layout(xaxis_rangeslider_visible=False)
figure.update_xaxes(
rangebreaks=[
    dict(bounds=["sat", "mon"])]
)
figure.show()


In [ ]:
df_test=df.iloc[:50]

In [ ]:
pos=[]

for i in tqdm(range(len(df_test))):

    temp_df=df_test.iloc[:i+1]

    fib_df=Fibonacci(temp_df)
    fib_df.calc_indicator(0.236,20)
    fib_df.calc_position()

    

    pos.append(fib_df.data['pos_trend_retr_0.236_per_20'].iloc[-1])

    
    
    

    

In [ ]:
pos

In [ ]:
fib_df_1=Fibonacci(df_test)

In [ ]:
fib_df_1.calc_indicator(0.236,20)

In [ ]:
fib_df_1.data.iloc[:50]

In [ ]:
pos==fib_df_1.data['pos_trend_retr_0.236_per_20'].iloc[:50]

In [ ]:
fib_df_1.data.iloc[:50]

In [ ]:
df_test['hh_right']=df_test['h'].rolling(5).apply(calc_hh, engine='numba', raw=True)

In [ ]:
df_test['hh_left']=df_test['h'].rolling(5, closed='left').apply(calc_hh, engine='numba', raw=True)

In [ ]:
df_test